<a href="https://colab.research.google.com/github/manassawant607-arch/gradio/blob/main/Copy_of_AI_DRP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [39]:
!pip install kaggle

In [2]:
from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username":"manash07","key":"4353793e822c50858b9695ebd8abbd0c"}'}

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!unzip -q tuberculosis-tb-chest-xray-dataset.zip

unzip:  cannot find or open tuberculosis-tb-chest-xray-dataset.zip, tuberculosis-tb-chest-xray-dataset.zip.zip or tuberculosis-tb-chest-xray-dataset.zip.ZIP.


In [5]:
!kaggle datasets download tawsifurrahman/tuberculosis-tb-chest-xray-dataset

Dataset URL: https://www.kaggle.com/datasets/tawsifurrahman/tuberculosis-tb-chest-xray-dataset
License(s): copyright-authors
100% 662M/663M [00:04<00:00, 86.5MB/s]
100% 663M/663M [00:04<00:00, 152MB/s] 


In [6]:
import os

# Ensure the dataset is unzipped before attempting to access its directory
# The previous unzip command failed, so we'll re-attempt it here if the directory doesn't exist.
if not os.path.exists("TB_Chest_Radiography_Database"):
    print("Unzipping tuberculosis-tb-chest-xray-dataset.zip...")
    !unzip -q tuberculosis-tb-chest-xray-dataset.zip
    # Optionally, verify if the directory was created after unzipping
    if not os.path.exists("TB_Chest_Radiography_Database"):
        print("Warning: 'TB_Chest_Radiography_Database' was not found after unzipping. Please check the dataset structure.")

dataset_path = "TB_Chest_Radiography_Database"

for folder in os.listdir(dataset_path):

    folder_path = os.path.join(dataset_path, folder)

    if os.path.isdir(folder_path):   # only process folders
        print(folder, len(os.listdir(folder_path)))

Unzipping tuberculosis-tb-chest-xray-dataset.zip...
Normal 3500
Tuberculosis 700


In [7]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

dataset_path = "TB_Chest_Radiography_Database"

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    dataset_path,
    target_size=(224,224),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Found 3360 images belonging to 2 classes.
Found 840 images belonging to 2 classes.


In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

model = Sequential([

    Conv2D(32,(3,3),activation='relu',input_shape=(224,224,3)),
    MaxPooling2D(2,2),

    Conv2D(64,(3,3),activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128,(3,3),activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),

    Dense(128,activation='relu'),
    Dense(1,activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,089 (42.61 MB)

 Trainable params: 11,169,089 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

In [9]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 49s 403ms/step - accuracy: 0.8938 - loss: 0.2648 - val_accuracy: 0.8726 - val_loss: 0.2261
Epoch 2/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 37s 349ms/step - accuracy: 0.9411 - loss: 0.1555 - val_accuracy: 0.9143 - val_loss: 0.2157
Epoch 3/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 39s 370ms/step - accuracy: 0.9670 - loss: 0.0839 - val_accuracy: 0.9464 - val_loss: 0.1465
Epoch 4/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 39s 368ms/step - accuracy: 0.9836 - loss: 0.0512 - val_accuracy: 0.9679 - val_loss: 0.1048
Epoch 5/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 37s 353ms/step - accuracy: 0.9899 - loss: 0.0303 - val_accuracy: 0.9583 - val_loss: 0.1069


In [10]:
model.save("tb_detection_model.h5")

In [11]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

In [12]:
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(128, activation="relu")(x)

predictions = Dense(1, activation="sigmoid")(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,422,081 (9.24 MB)

 Trainable params: 164,097 (641.00 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [13]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

Epoch 1/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 74s 530ms/step - accuracy: 0.9679 - loss: 0.0917 - val_accuracy: 0.9940 - val_loss: 0.0238
Epoch 2/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 37s 349ms/step - accuracy: 0.9952 - loss: 0.0180 - val_accuracy: 0.9952 - val_loss: 0.0133
Epoch 3/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 38s 359ms/step - accuracy: 0.9976 - loss: 0.0077 - val_accuracy: 0.9952 - val_loss: 0.0185
Epoch 4/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 38s 359ms/step - accuracy: 0.9985 - loss: 0.0048 - val_accuracy: 0.9988 - val_loss: 0.0077
Epoch 5/5
105/105 ━━━━━━━━━━━━━━━━━━━━ 37s 351ms/step - accuracy: 0.9997 - loss: 0.0024 - val_accuracy: 0.9988 - val_loss: 0.0089


In [14]:
model.save("tb_detector_ai.h5")

In [15]:
!pip install kaggle

In [16]:
!kaggle datasets download -d khushikyad001/tuberculosis-trends-global-and-regional-insights

Dataset URL: https://www.kaggle.com/datasets/khushikyad001/tuberculosis-trends-global-and-regional-insights
License(s): MIT
  0% 0.00/159k [00:00<?, ?B/s]
100% 159k/159k [00:00<00:00, 602MB/s]


In [17]:
!unzip tuberculosis-trends-global-and-regional-insights.zip

Archive:  tuberculosis-trends-global-and-regional-insights.zip
  inflating: Tuberculosis_Trends.csv  


In [18]:
import os
os.listdir()

['.config',
 'tuberculosis-trends-global-and-regional-insights.zip',
 'tb_detection_model.h5',
 'drive',
 'tuberculosis-tb-chest-xray-dataset.zip',
 'TB_Chest_Radiography_Database',
 'kaggle.json',
 'Tuberculosis_Trends.csv',
 'tb_detector_ai.h5',
 'sample_data']

In [19]:
import pandas as pd

df = pd.read_csv("Tuberculosis_Trends.csv")

df.head()

,Country,Region,Income_Level,Year,TB_Cases,TB_Deaths,TB_Incidence_Rate,TB_Mortality_Rate,TB_Treatment_Success_Rate,Drug_Resistant_TB_Cases,...,GDP_Per_Capita,Health_Expenditure_Per_Capita,Urban_Population_Percentage,Malnutrition_Prevalence,Smoking_Prevalence,TB_Doctors_Per_100K,TB_Hospitals_Per_Million,Access_To_Health_Services,BCG_Vaccination_Coverage,HIV_Testing_Coverage
0,Indonesia,Asia,Upper-Middle,2002,14128,3846,150.58,18.99,54.52,3977,...,25516,3180,50.49,28.84,22.90,4.10,15.46,52.38,95.87,88.39
1,Brazil,North America,Lower-Middle,2017,1069,342,167.80,44.20,88.19,4179,...,26300,5529,24.06,28.65,33.79,8.18,3.06,46.95,63.78,35.60
2,Nigeria,South America,High,2006,2473,6316,285.77,6.52,64.55,4961,...,30771,1703,81.77,38.69,6.63,2.26,4.95,40.88,68.91,77.81
3,Russia,Africa,Lower-Middle,2014,36090,6420,146.46,24.86,69.77,4012,...,35730,8354,42.19,20.38,15.00,8.33,13.68,62.20,71.14,84.91
4,Indonesia,Europe,Upper-Middle,2020,33538,4910,56.87,10.52,64.57,2342,...,38391,8024,64.97,39.28,18.00,6.60,4.35,53.88,55.11,49.89


In [20]:
print(df.columns.tolist())

['Country', 'Region', 'Income_Level', 'Year', 'TB_Cases', 'TB_Deaths', 'TB_Incidence_Rate', 'TB_Mortality_Rate', 'TB_Treatment_Success_Rate', 'Drug_Resistant_TB_Cases', 'HIV_CoInfected_TB_Cases', 'Population', 'GDP_Per_Capita', 'Health_Expenditure_Per_Capita', 'Urban_Population_Percentage', 'Malnutrition_Prevalence', 'Smoking_Prevalence', 'TB_Doctors_Per_100K', 'TB_Hospitals_Per_Million', 'Access_To_Health_Services', 'BCG_Vaccination_Coverage', 'HIV_Testing_Coverage']


In [21]:
X = df[["TB_Cases","TB_Deaths","TB_Treatment_Success_Rate"]]

y = df["TB_Incidence_Rate"]

In [22]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

res_model = RandomForestRegressor()

res_model.fit(X_train, y_train)

RandomForestRegressor()

In [23]:
!pip install gradio

In [24]:
import gradio as gr
import numpy as np
import tensorflow as tf
from PIL import Image

In [25]:
tb_model = tf.keras.models.load_model("tb_detection_model.h5")

In [26]:
def preprocess(img):
    img = img.resize((224,224))
    img = np.array(img)/255.0
    img = np.expand_dims(img, axis=0)
    return img

In [27]:
def predict_tb(image):

    img = preprocess(image)

    pred = tb_model.predict(img)[0][0]

    if pred > 0.5:
        tb_result = "TB Detected"
    else:
        tb_result = "Normal"

    # Example resistance prediction
    sample = [[20000,2000,85]]
    res_pred = res_model.predict(sample)[0]

    return f"X-ray Result: {tb_result}\nPredicted TB incidence risk: {res_pred}"

In [28]:
app = gr.Interface(
    fn=predict_tb,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="AI TB Detection + Resistance Prediction"
)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c87d9ab0aee86d9b3e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [29]:
def predict_tb(image):

    img = preprocess(image)
    pred = tb_model.predict(img)[0][0]

    # Step 1 TB detection
    if pred > 0.5:
        tb_result = "TB Detected"
    else:
        tb_result = "Normal"

    # Step 2 Mutation Analysis (prototype)
    mutation = "rpoB mutation detected"

    # Step 3 Drug Resistance Prediction
    if "rpoB" in mutation:
        resistance = "Rifampicin Resistant (Possible MDR-TB)"
    else:
        resistance = "Drug Sensitive TB"

    # Step 4 Treatment Recommendation
    if resistance == "Rifampicin Resistant (Possible MDR-TB)":
        treatment = "Recommended: Bedaquiline + Linezolid + Levofloxacin regimen"
    else:
        treatment = "Recommended: Isoniazid + Rifampicin + Pyrazinamide + Ethambutol"

    return tb_result, mutation, resistance, treatment

In [30]:
app = gr.Interface(
    fn=predict_tb,
    inputs=gr.Image(type="pil", label="Upload Chest X-ray"),
    outputs=[
        gr.Textbox(label="TB Detection"),
        gr.Textbox(label="Mutation Analysis"),
        gr.Textbox(label="Drug Resistance Prediction"),
        gr.Textbox(label="Treatment Recommendation")
    ],
    title="AI TB Detection + Drug Resistance + Treatment System"
)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://edf6768014ac9343a8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [31]:
import gradio as gr
import numpy as np
import tensorflow as tf
from PIL import Image

# load TB detection model
model = tf.keras.models.load_model("tb_detection_model.h5")

def preprocess(img):
    img = img.resize((224,224))
    img = np.array(img)/255.0
    img = np.expand_dims(img, axis=0)
    return img

def predict_tb(image):

    img = preprocess(image)

    pred = model.predict(img)[0][0]

    # TB detection
    if pred > 0.5:
        tb = "TB Detected"
        mutation = "rpoB mutation detected"
        resistance = "Rifampicin Resistant (Possible MDR-TB)"
        treatment = "Bedaquiline + Linezolid + Levofloxacin"
    else:
        tb = "Normal"
        mutation = "No mutation detected"
        resistance = "Drug Sensitive"
        treatment = "Standard TB Therapy"

    return tb, mutation, resistance, treatment


app = gr.Interface(
    fn=predict_tb,
    inputs=gr.Image(type="pil", label="Upload Chest X-ray"),
    outputs=[
        gr.Textbox(label="TB Detection"),
        gr.Textbox(label="Mutation Analysis"),
        gr.Textbox(label="Drug Resistance Prediction"),
        gr.Textbox(label="Treatment Recommendation")
    ],
    title="AI TB Detection + Drug Resistance + Treatment System"
)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f8d2461aebd7f7b826.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [32]:
!ls


drive
kaggle.json
sample_data
TB_Chest_Radiography_Database
tb_detection_model.h5
tb_detector_ai.h5
tuberculosis-tb-chest-xray-dataset.zip
Tuberculosis_Trends.csv
tuberculosis-trends-global-and-regional-insights.zip


In [33]:
from google.colab import files
files.download("tb_detection_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [35]:
from tensorflow.keras.models import load_model

model = load_model("/content/tb_detection_model.h5")

In [36]:
import gradio as gr
import numpy as np
import tensorflow as tf
from PIL import Image

model = tf.keras.models.load_model("tb_detection_model.h5")

def preprocess(img):
    img = img.resize((224,224))
    img = np.array(img)/255.0
    img = np.expand_dims(img, axis=0)
    return img

def predict_tb(image):

    img = preprocess(image)
    pred = model.predict(img)[0][0]

    if pred > 0.5:
        tb = "TB Detected"
        mutation = "rpoB mutation detected"
        resistance = "Rifampicin Resistant (Possible MDR-TB)"
        treatment = "Bedaquiline + Linezolid + Levofloxacin"
    else:
        tb = "Normal"
        mutation = "No mutation detected"
        resistance = "Drug Sensitive"
        treatment = "Standard TB therapy"

    return tb, mutation, resistance, treatment


demo = gr.Interface(
    fn=predict_tb,
    inputs=gr.Image(type="pil", label="Upload Chest X-ray"),
    outputs=[
        gr.Textbox(label="TB Detection"),
        gr.Textbox(label="Mutation Analysis"),
        gr.Textbox(label="Drug Resistance Prediction"),
        gr.Textbox(label="Treatment Recommendation")
    ],
    title="AI TB Detection + Drug Resistance + Treatment System"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d653e45e9ce4a0a95e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [37]:
import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image

model = tf.keras.models.load_model("tb_detection_model.h5")

def predict(image):
    img = image.resize((224,224))
    img = np.array(img)/255.0
    img = np.expand_dims(img, axis=0)

    prediction = model.predict(img)[0][0]

    if prediction > 0.5:
        return "⚠️ TB Detected"
    else:
        return "✅ Normal"

interface = gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="AI TB Chest X-ray Detector",
    description="Upload a chest X-ray to detect Tuberculosis using AI."
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c2670e1e06adea4ca5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [38]:
import gradio as gr
import numpy as np
import tensorflow as tf
from PIL import Image

model = tf.keras.models.load_model("tb_detection_model.h5")

def preprocess(img):
    img = img.resize((224,224))
    img = np.array(img)/255.0
    img = np.expand_dims(img, axis=0)
    return img

def predict_tb(image):

    img = preprocess(image)
    pred = model.predict(img)[0][0]

    if pred > 0.5:
        tb = "TB Detected"
        mutation = "rpoB mutation detected"
        resistance = "Rifampicin Resistant (Possible MDR-TB)"
        treatment = "Bedaquiline + Linezolid + Levofloxacin"
    else:
        tb = "Normal"
        mutation = "No mutation detected"
        resistance = "Drug Sensitive"
        treatment = "Standard TB therapy"

    return tb, mutation, resistance, treatment


demo = gr.Interface(
    fn=predict_tb,
    inputs=gr.Image(type="pil", label="Upload Chest X-ray"),
    outputs=[
        gr.Textbox(label="TB Detection"),
        gr.Textbox(label="Mutation Analysis"),
        gr.Textbox(label="Drug Resistance Prediction"),
        gr.Textbox(label="Treatment Recommendation")
    ],
    title="AI TB Detection + Drug Resistance + Treatment System"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://334181e906e5e8dc8c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
